# 05 — Diccionario de métricas

Este notebook documenta de forma exhaustiva las **19 variables** del dataset final `Matriz_V3.xlsx`, organizadas en cuatro categorías funcionales. Para cada variable se detallan el tipo, las unidades, el rango observado, la fuente de origen y las transformaciones aplicadas durante el pipeline de limpieza (notebooks 01–04).

**Dataset de referencia:** `Matriz_V3.xlsx` — resultado del pipeline completo de limpieza.

**Notebooks previos:**
- `01_limpieza_previa.ipynb` → limpieza de negocio (duplicados, NaN, IDs)
- `02_seleccion_vd.ipynb` → selección y justificación de las 4 VD
- `03_creacion_metricas.ipynb` → métricas derivadas, normalización y columna Equipo
- `04_limpieza_metricas.ipynb` → exclusión estadística de no-participación

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ── Carga del dataset final ──
df = pd.read_excel("../Datos/Matriz_V3.xlsx")
N = len(df)
print(f"✅ Dataset cargado — {N:,} filas × {df.shape[1]} columnas")

# ── Función auxiliar para estadísticos formateados ──
def stats_num(col_name):
    """Devuelve un diccionario con estadísticos de una columna numérica."""
    s = pd.to_numeric(df[col_name], errors="coerce").dropna()
    q25, q75 = s.quantile(0.25), s.quantile(0.75)
    return {
        "n": len(s),
        "min": s.min(),
        "max": s.max(),
        "media": s.mean(),
        "sd": s.std(),
        "mediana": s.median(),
        "q25": q25,
        "q75": q75,
        "iqr": q75 - q25,
        "skew": s.skew(),
        "pct_ceros": (s == 0).sum() / len(s) * 100,
        "pct_missing": df[col_name].isna().sum() / N * 100,
    }


def stats_cat(col_name):
    """Devuelve un diccionario con estadísticos de una columna categórica."""
    s = df[col_name]
    vc = s.value_counts(dropna=False)
    return {
        "n_validos": s.notna().sum(),
        "n_niveles": s.nunique(),
        "pct_missing": s.isna().sum() / N * 100,
        "distribucion": vc,
    }


print(f"\nColumnas ({df.shape[1]}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}  ({df[col].dtype})")

✅ Dataset cargado — 4,446 filas × 19 columnas

Columnas (19):
   1. Phase Id  (int64)
   2. Agrupacion  (str)
   3. Espacio  (str)
   4. Polaridad  (str)
   5. Equilibrio  (str)
   6. NombreCorrecto  (str)
   7. GrupoEdad  (str)
   8. Phase Duration (min)  (int64)
   9. Player Id  (int64)
  10. Position Category  (str)
  11. Position  (str)
  12. Total Touches (#)  (int64)
  13. Total Touches / min  (float64)
  14. Golpeos +15 m/s  (int64)
  15. Golpeos +15 m/s / min  (float64)
  16. Distance Covered (m)  (int64)
  17. Distance Covered (m) / min  (float64)
  18. HID Covered (m) Zone 1 [> 4(m/s)]  (int64)
  19. HID Covered (m) / min  (float64)


---

## 1. Métricas derivadas — Variables dependientes normalizadas

Son las variables respuesta principales del estudio. Se obtienen dividiendo cada VD absoluta por `Phase Duration (min)`, generando **tasas por minuto** que permiten comparar tareas de distinta duración. Todas fueron creadas en `03_creacion_metricas.ipynb`.

In [ ]:
# ── Estadísticos de las 4 VD normalizadas ──
vd_norm = {
    "Total Touches / min":                          stats_num("Total Touches / min"),
    "Golpeos +15 m/s / min":                        stats_num("Golpeos +15 m/s / min"),
    "Distance Covered (m) / min":                   stats_num("Distance Covered (m) / min"),
    "High Intensity Distance (20 km/h) / min":      stats_num("High Intensity Distance (20 km/h) / min"),
}

for nombre, st in vd_norm.items():
    print(f"\n{'─'*60}")
    print(f"  {nombre}")
    print(f"{'─'*60}")
    print(f"  Rango observado: [{st['min']:.2f} – {st['max']:.2f}]")
    print(f"  Media ± SD:      {st['media']:.2f} ± {st['sd']:.2f}")
    print(f"  Mediana (IQR):   {st['mediana']:.2f} ({st['q25']:.2f} – {st['q75']:.2f})")
    print(f"  Asimetría:       {st['skew']:.2f}")
    print(f"  Ceros:           {st['pct_ceros']:.1f}%")
    print(f"  Missings:        {st['pct_missing']:.1f}%")


────────────────────────────────────────────────────────────
  Total Touches / min
────────────────────────────────────────────────────────────
  Rango observado: [0.10 – 14.42]
  Media ± SD:      2.49 ± 1.78
  Mediana (IQR):   2.04 (1.27 – 3.22)
  Asimetría:       1.77
  Ceros:           0.0%
  Missings:        0.0%

────────────────────────────────────────────────────────────
  Golpeos +15 m/s / min
────────────────────────────────────────────────────────────
  Rango observado: [0.00 – 4.50]
  Media ± SD:      0.16 ± 0.22
  Mediana (IQR):   0.11 (0.00 – 0.23)
  Asimetría:       4.83
  Ceros:           29.5%
  Missings:        0.0%

────────────────────────────────────────────────────────────
  Distance Covered (m) / min
────────────────────────────────────────────────────────────
  Rango observado: [7.70 – 172.00]
  Media ± SD:      79.17 ± 28.21
  Mediana (IQR):   78.38 (57.73 – 99.91)
  Asimetría:       0.10
  Ceros:           0.0%
  Missings:        0.0%

────────────────────────

### 1.1. `Total Touches / min` — Toques por minuto

| Atributo | Valor |
|---|---|
| **Rol** | Variable dependiente (VD₁ normalizada) |
| **Tipo** | Cuantitativa continua |
| **Unidad** | toques / minuto |
| **Fuente** | Derivada — calculada en NB03 a partir de `Total Touches (#)` / `Phase Duration (min)` |
| **Transformación** | `Total Touches / min = Total Touches (#) / Phase Duration (min)` |

**Justificación de la normalización:**  
El número absoluto de toques (`Total Touches (#)`) depende linealmente de la duración de la tarea. Un jugador en una tarea de 30 minutos acumulará mecánicamente más toques que otro en una tarea de 5 minutos, con independencia de su rendimiento real. La normalización por `Phase Duration (min)` convierte el conteo absoluto en una **tasa de intensidad técnica** comparable entre tareas de distinta duración.

**Papel en la limpieza (NB04):**  
Se utilizó junto con `Total Touches (#)` como criterio de exclusión de no-participación. Las instancias con AMBAS métricas por debajo del percentil 5 fueron eliminadas (criterio AND conservador).

### 1.2. `Golpeos +15 m/s / min` — Golpeos potentes por minuto

| Atributo | Valor |
|---|---|
| **Rol** | Variable dependiente (VD₂ normalizada) |
| **Tipo** | Cuantitativa continua |
| **Unidad** | golpeos / minuto |
| **Fuente** | Derivada — calculada en NB03 a partir de `Golpeos +15 m/s` / `Phase Duration (min)` |
| **Transformación** | `Golpeos +15 m/s / min = Golpeos +15 m/s / Phase Duration (min)` |

**Nota sobre los ceros:**  
Esta variable presenta una **alta proporción natural de ceros**: muchos registros corresponden a tareas o jugadores que no ejecutaron golpeos a más de 15 m/s durante la fase. Estos ceros son legítimos y no indican participación residual. Por este motivo, `Golpeos +15 m/s` **no se utilizó como criterio de exclusión** en NB04.

**Composición de la métrica base:**  
`Golpeos +15 m/s` se creó en NB03 como la **suma** de las zonas de velocidad de release superiores a 15 m/s:  
`RV Zone 4 [15–20 m/s] + RV Zone 5 [20–25 m/s] + RV Zone 6 [> 25 m/s]`

### 1.3. `Distance Covered (m) / min` — Distancia por minuto

| Atributo | Valor |
|---|---|
| **Rol** | Variable dependiente (VD₃ normalizada) |
| **Tipo** | Cuantitativa continua |
| **Unidad** | metros / minuto |
| **Fuente** | Derivada — calculada en NB03 a partir de `Distance Covered (m)` / `Phase Duration (min)` |
| **Transformación** | `Distance Covered (m) / min = Distance Covered (m) / Phase Duration (min)` |

**Interpretación:**  
Representa la **intensidad relativa de desplazamiento**. Valores altos indican tareas con gran demanda locomotora (transiciones, rondos amplios), mientras que valores bajos sugieren tareas más estáticas o técnicas.

**Papel en la validación (NB04):**  
Se analizó como **validación cruzada** del criterio de exclusión basado en toques. Las instancias eliminadas por el criterio de toques también presentaron valores muy bajos en Distance/min, lo que confirmó la coherencia del filtrado. No se aplicó filtro independiente por distancia.

### 1.4. `High Intensity Distance (20 km/h) / min` — Distancia a alta intensidad por minuto

| Atributo | Valor |
|---|---|
| **Rol** | Variable dependiente (VD₄ normalizada) |
| **Tipo** | Cuantitativa continua |
| **Unidad** | metros / minuto |
| **Fuente** | Derivada — calculada en NB03 a partir de `High Intensity Distance (20 km/h)` / `Phase Duration (min)` |
| **Transformación** | `High Intensity Distance (20 km/h) / min = High Intensity Distance (20 km/h) / Phase Duration (min)` |

**Nota sobre los ceros:**  
Los ceros son legítimos y frecuentes en tareas de carácter predominantemente técnico (rondos, ejercicios analíticos), donde los desplazamientos a alta intensidad (> 5.5 m/s ≈ 20 km/h) son escasos o nulos. Por este motivo, **no se utilizó como criterio de exclusión** en NB04.

**Complementariedad con Distance Covered:**  
Mientras que `Distance Covered (m) / min` captura el **volumen total de carga locomotora**, `High Intensity Distance (20 km/h) / min` mide específicamente la **intensidad del esfuerzo en sprint**. Ambas son complementarias (r ≈ 0.53 entre sus versiones absolutas), con menor redundancia que la que presentaba la anterior métrica HID (umbral > 4 m/s, r ≈ 0.81).

---

## 2. Métricas base — Variables dependientes absolutas

Son las variables originales del dispositivo PlayerMaker (o derivadas a partir de ellas) que representan conteos o magnitudes totales acumuladas durante la fase de entrenamiento. Sirven como **numeradores** de las VD normalizadas y, en el caso de `Total Touches (#)`, como **criterio auxiliar de exclusión**.

In [ ]:
# ── Estadísticos de las 4 VD absolutas ──
vd_abs = {
    "Total Touches (#)":                    stats_num("Total Touches (#)"),
    "Golpeos +15 m/s":                      stats_num("Golpeos +15 m/s"),
    "Distance Covered (m)":                 stats_num("Distance Covered (m)"),
    "High Intensity Distance (20 km/h)":    stats_num("High Intensity Distance (20 km/h)"),
}

for nombre, st in vd_abs.items():
    print(f"\n{'─'*60}")
    print(f"  {nombre}")
    print(f"{'─'*60}")
    print(f"  Rango observado: [{st['min']:.0f} – {st['max']:.0f}]")
    print(f"  Media ± SD:      {st['media']:.1f} ± {st['sd']:.1f}")
    print(f"  Mediana (IQR):   {st['mediana']:.0f} ({st['q25']:.0f} – {st['q75']:.0f})")
    print(f"  Asimetría:       {st['skew']:.2f}")
    print(f"  Ceros:           {st['pct_ceros']:.1f}%")
    print(f"  Missings:        {st['pct_missing']:.1f}%")


────────────────────────────────────────────────────────────
  Total Touches (#)
────────────────────────────────────────────────────────────
  Rango observado: [1 – 173]
  Media ± SD:      35.5 ± 25.3
  Mediana (IQR):   29 (17 – 47)
  Asimetría:       1.45
  Ceros:           0.0%
  Missings:        0.0%

────────────────────────────────────────────────────────────
  Golpeos +15 m/s
────────────────────────────────────────────────────────────
  Rango observado: [0 – 38]
  Media ± SD:      2.4 ± 3.0
  Mediana (IQR):   2 (0 – 3)
  Asimetría:       2.97
  Ceros:           29.5%
  Missings:        0.0%

────────────────────────────────────────────────────────────
  Distance Covered (m)
────────────────────────────────────────────────────────────
  Rango observado: [35 – 4442]
  Media ± SD:      1211.0 ± 690.7
  Mediana (IQR):   1087 (727 – 1539)
  Asimetría:       1.18
  Ceros:           0.0%
  Missings:        0.0%

────────────────────────────────────────────────────────────
  HID Cover

### 2.1. `Total Touches (#)` — Número total de toques

| Atributo | Valor |
|---|---|
| **Rol** | VD absoluta + criterio de exclusión (NB04) |
| **Tipo** | Cuantitativa discreta |
| **Unidad** | toques (conteo absoluto) |
| **Fuente** | Directa — PlayerMaker (acelerómetros en las botas) |
| **Transformación** | Numerador de `Total Touches / min`; criterio de exclusión (p05 conjunto) |

**Doble función:**  
1. **Numerador** de la VD normalizada `Total Touches / min`.
2. **Criterio de exclusión** en NB04: se utilizó junto con `Total Touches / min` para identificar instancias de no-participación. Solo se eliminaron las que caían por debajo del percentil 5 en **ambas** métricas simultáneamente (criterio AND).

**Selección como VD (NB02):**  
Se seleccionó como proxy global de actividad técnica. Presenta correlación alta con `Releases (#)` (r ≈ 0.88), lo que permite subsumirla: un jugador con muchos toques también ejecuta muchos releases, y viceversa.

### 2.2. `Golpeos +15 m/s` — Golpeos potentes (total)

| Atributo | Valor |
|---|---|
| **Rol** | VD absoluta (numerador de VD₂ normalizada) |
| **Tipo** | Cuantitativa discreta |
| **Unidad** | golpeos (conteo absoluto) |
| **Fuente** | Derivada — creada en NB03 como suma de zonas de velocidad de release |
| **Transformación** | `Golpeos +15 m/s = RV Zone 4 [15-20 m/s] + RV Zone 5 [20-25 m/s] + RV Zone 6 [> 25 m/s]` |

**Selección como VD (NB02):**  
Seleccionada como indicador de **potencia/calidad de golpeo**. Presenta baja correlación con `Total Touches (#)` (r ≈ 0.33), lo que la convierte en una dimensión complementaria: un jugador puede tocar mucho el balón sin ejecutar golpeos potentes, y viceversa.

**Columnas eliminadas tras la creación:**  
Las tres zonas de velocidad originales (`RV Zone 4`, `5`, `6`) se eliminaron del dataset final por redundancia, al quedar recogidas en esta métrica agregada.

### 2.3. `Distance Covered (m)` — Distancia total recorrida

| Atributo | Valor |
|---|---|
| **Rol** | VD absoluta (numerador de VD₃ normalizada) |
| **Tipo** | Cuantitativa discreta |
| **Unidad** | metros |
| **Fuente** | Directa — PlayerMaker (GPS/acelerómetro) |
| **Transformación** | Ninguna sobre la variable; usada como numerador de `Distance Covered (m) / min` |

**Selección como VD (NB02):**  
Seleccionada como indicador del **volumen de carga física total**. En el dataset final (tras la limpieza de NB04) la proporción de ceros es 0.0 %, ya que las pocas instancias con distancia nula fueron eliminadas por el criterio de toques.

**Análisis en NB04:**  
Se verificó que las 6 instancias con `Distance Covered = 0` quedaban cubiertas por el criterio de exclusión de toques, sin necesidad de un filtro independiente por distancia.

### 2.4. `High Intensity Distance (20 km/h)` — Distancia a alta intensidad

| Atributo | Valor |
|---|---|
| **Rol** | VD absoluta (numerador de VD₄ normalizada) |
| **Tipo** | Cuantitativa discreta |
| **Unidad** | metros |
| **Fuente** | Directa — PlayerMaker (GPS/acelerómetro), umbral de velocidad > 5.5 m/s (≈ 20 km/h). Variable original: `SD Covered (m) [> 5.5(m/s)]`, renombrada en NB03. |
| **Transformación** | Renombrada en NB03: `SD Covered (m) [> 5.5(m/s)]` → `High Intensity Distance (20 km/h)`. Usada como numerador de `High Intensity Distance (20 km/h) / min` |

**Selección como VD (NB02):**  
Seleccionada como indicador de **intensidad de carga física**. El umbral de 5.5 m/s (≈ 19.8 km/h) se ajusta a lo que la literatura científica considera carrera de alta intensidad en fútbol (Akenhead & Nassis, 2016; Bradley et al., 2009). Presenta una correlación moderada con `Distance Covered (m)` (r ≈ 0.65), menor que la de la antigua variable HID (umbral > 4 m/s, r ≈ 0.81), lo que reduce la redundancia entre ambas métricas físicas.

**Ceros legítimos:**  
La proporción de ceros es elevada y legítima: en tareas técnicas o de espacio reducido, los jugadores pueden no alcanzar velocidades de sprint (> 5.5 m/s). Estos ceros **no fueron tratados como outliers** en NB04 — representan la ausencia real de esfuerzo a alta intensidad en la tarea.

---

## 3. Variables de diseño de tarea

Son las variables independientes (VI) que describen las **condiciones de la tarea de entrenamiento**. Permiten segmentar las observaciones para analizar su efecto sobre las VD. Todas fueron codificadas manualmente por el investigador y verificadas en NB01.

In [10]:
# ── Estadísticos de las variables de diseño de tarea ──
vi_tarea = ["Agrupacion", "Espacio", "Polaridad", "Equilibrio"]

for col_name in vi_tarea:
    st = stats_cat(col_name)
    print(f"\n{'─'*60}")
    print(f"  {col_name}  ({st['n_niveles']} niveles)")
    print(f"{'─'*60}")
    for nivel, n in st["distribucion"].items():
        pct = n / N * 100
        print(f"    {str(nivel):25s}  {n:>5,}  ({pct:>5.1f}%)")
    print(f"  Missings: {st['pct_missing']:.1f}%")


────────────────────────────────────────────────────────────
  Agrupacion  (2 niveles)
────────────────────────────────────────────────────────────
    grande                     2,293  ( 51.6%)
    pequeno                    2,153  ( 48.4%)
  Missings: 0.0%

────────────────────────────────────────────────────────────
  Espacio  (2 niveles)
────────────────────────────────────────────────────────────
    reducido                   2,294  ( 51.6%)
    amplio                     2,152  ( 48.4%)
  Missings: 0.0%

────────────────────────────────────────────────────────────
  Polaridad  (2 niveles)
────────────────────────────────────────────────────────────
    Polarizado                 3,477  ( 78.2%)
    No_polarizado                969  ( 21.8%)
  Missings: 0.0%

────────────────────────────────────────────────────────────
  Equilibrio  (2 niveles)
────────────────────────────────────────────────────────────
    Equilibrio                 3,346  ( 75.3%)
    Desequilibrio           

### 3.1. `Agrupacion` — Tamaño de agrupación de la tarea

| Atributo | Valor |
|---|---|
| **Rol** | Variable independiente (VI₁) |
| **Tipo** | Categórica nominal (2 niveles) |
| **Unidad** | — (categoría) |
| **Fuente** | Codificación manual del investigador |
| **Transformación** | NB01: verificada consistencia intra-Phase Id; reclasificación de las 7 combinaciones originales Agrupación×Espacio en 4 categorías simplificadas → 2 niveles de agrupación (`pequeno`, `grande`) |

**Interpretación de los niveles:**  
- **pequeno**: tareas con grupos pequeños de jugadores (menor número de participantes por equipo).  
- **grande**: tareas con grupos grandes de jugadores (mayor número de participantes por equipo).  

La agrupación modifica la frecuencia de contacto con el balón y las oportunidades de interacción técnica. Las combinaciones Agrupación×Espacio ambiguas (AMedia×EMedio, APequena×EMedio) fueron eliminadas en NB01 por no ser clasificables de forma unívoca.

### 3.2. `Espacio` — Tipo de espacio de la tarea

| Atributo | Valor |
|---|---|
| **Rol** | Variable independiente (VI₂) |
| **Tipo** | Categórica nominal (2 niveles) |
| **Unidad** | — (categoría) |
| **Fuente** | Codificación manual del investigador |
| **Transformación** | NB01: verificada consistencia intra-Phase Id; reclasificación de las 7 combinaciones originales Agrupación×Espacio en 4 categorías simplificadas → 2 niveles de espacio (`reducido`, `amplio`) |

**Interpretación de los niveles:**  
- **reducido**: tareas en espacios de juego de dimensiones pequeñas, que favorecen el contacto con el balón y la proximidad entre jugadores.  
- **amplio**: tareas en espacios de juego de dimensiones grandes, que favorecen las transiciones y el juego posicional.

### 3.3. `Polaridad` — Polaridad de la tarea

| Atributo | Valor |
|---|---|
| **Rol** | Variable independiente (VI₃) |
| **Tipo** | Categórica nominal |
| **Unidad** | — (categoría) |
| **Fuente** | Codificación manual del investigador |
| **Transformación** | NB01: verificada consistencia intra-Phase Id |

**Interpretación de los niveles:**  
- **Polarizado**: tarea con roles asimétricos (un equipo ataca, otro defiende), con orientación clara hacia una portería. Favorece situaciones de juego direccional.
- **No polarizado**: tarea sin dirección preferente de ataque, típicamente rondos, juegos de posesión o ejercicios simétricos.

### 3.4. `Equilibrio` — Equilibrio numérico de la tarea

| Atributo | Valor |
|---|---|
| **Rol** | Variable independiente (VI₄) |
| **Tipo** | Categórica nominal |
| **Unidad** | — (categoría) |
| **Fuente** | Codificación manual del investigador |
| **Transformación** | NB01: verificada consistencia intra-Phase Id |

**Interpretación de los niveles:**  
- **Equilibrio**: tarea con el mismo número de jugadores en ambos equipos (ej. 5v5, 8v8).
- **Desequilibrio**: tarea con superioridad/inferioridad numérica (ej. 5v3, 4v4+2 comodines). El desequilibrio modifica la frecuencia de toques y las oportunidades de acción técnica.

---

## 4. Variables de contexto e identificación

Son las variables que identifican unívocamente cada observación (jugador × fase) y proporcionan el contexto necesario para la interpretación: equipo, posición, duración de la tarea.

In [12]:
# ── Estadísticos de las variables de contexto e identificación ──
vi_contexto_cat = ["NombreCorrecto", "GrupoEdad", "Position Category", "Position"]
vi_contexto_num = ["Phase Id", "Player Id", "Phase Duration (min)"]

print("═" * 60)
print("  VARIABLES CATEGÓRICAS DE CONTEXTO")
print("═" * 60)
for col_name in vi_contexto_cat:
    if col_name not in df.columns:
        print(f"\n  ⚠️  '{col_name}' no encontrada en el dataset")
        continue
    st = stats_cat(col_name)
    print(f"\n{'─'*60}")
    print(f"  {col_name}  ({st['n_niveles']} niveles)")
    print(f"{'─'*60}")
    for nivel, n in st["distribucion"].items():
        pct = n / N * 100
        print(f"    {str(nivel):30s}  {n:>5,}  ({pct:>5.1f}%)")
    print(f"  Missings: {st['pct_missing']:.1f}%")

print(f"\n\n{'═'*60}")
print("  VARIABLES NUMÉRICAS DE CONTEXTO")
print("═" * 60)
for col_name in vi_contexto_num:
    st = stats_num(col_name)
    print(f"\n{'─'*60}")
    print(f"  {col_name}")
    print(f"{'─'*60}")
    print(f"  Valores únicos:  {df[col_name].nunique():,}")
    print(f"  Rango:           [{st['min']:.0f} – {st['max']:.0f}]")
    print(f"  Media ± SD:      {st['media']:.1f} ± {st['sd']:.1f}")
    print(f"  Missings:        {st['pct_missing']:.1f}%")

════════════════════════════════════════════════════════════
  VARIABLES CATEGÓRICAS DE CONTEXTO
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  NombreCorrecto  (12 niveles)
────────────────────────────────────────────────────────────
    EASO                              533  ( 12.0%)
    Juvenil División de Honor         473  ( 10.6%)
    Neskak A                          442  (  9.9%)
    Neskak B                          439  (  9.9%)
    Infantil Txiki                    413  (  9.3%)
    Real Sociedad C                   400  (  9.0%)
    Cadete Vasca                      361  (  8.1%)
    Cadete Txiki                      355  (  8.0%)
    SANSE                             318  (  7.2%)
    Real Sociedad                     297  (  6.7%)
    Infantil Handi                    212  (  4.8%)
    Neskak C                          203  (  4.6%)
  Missings: 0.0%

──────────────────────────────────────────────

### 4.1. `Phase Id` — Identificador de fase

| Atributo | Valor |
|---|---|
| **Rol** | Identificador (nivel 2 de la jerarquía) |
| **Tipo** | Cuantitativa discreta (ID numérico) |
| **Unidad** | adimensional |
| **Fuente** | PlayerMaker (raw) |
| **Transformación** | NB01: eliminación de duplicados completos por `(Phase Id, Player Id)` |

**Interpretación:**  
Cada `Phase Id` identifica una **fase de entrenamiento** dentro de una sesión. Un mismo `Phase Id` puede contener múltiples jugadores (una fila por jugador). Es el segundo nivel de la estructura jerárquica: `Jugador → Fase → Sesión`.

### 4.2. `Player Id` — Identificador de jugador

| Atributo | Valor |
|---|---|
| **Rol** | Identificador (nivel 1 de la jerarquía) |
| **Tipo** | Cuantitativa discreta (ID numérico) |
| **Unidad** | adimensional |
| **Fuente** | PlayerMaker (raw) + unificación en NB01 |
| **Transformación** | NB01: unificación de Player Id para jugadores con múltiples registros (28 jugadores, se asigna ID mayoritario, 293 filas modificadas); eliminación de duplicados por `(Phase Id, Player Id)` |

**Nota sobre re-registros:**  
Algunos jugadores fueron registrados más de una vez en el sistema PlayerMaker, generando múltiples `Player Id` para la misma persona. En NB01 se unificaron asignando el ID mayoritario, con la excepción documentada de un nombre genérico que correspondía a 3 jugadores reales distintos.

### 4.3. `NombreCorrecto` — Nombre del equipo/categoría

| Atributo | Valor |
|---|---|
| **Rol** | Variable de identificación / agrupación |
| **Tipo** | Categórica nominal |
| **Unidad** | — (categoría) |
| **Fuente** | PlayerMaker (raw) — etiqueta legible unificada |
| **Transformación** | NB01: unificación de nombres tras resolución de IDs duplicados. NB03: utilizada como base para derivar `Equipo` |

**Interpretación:**  
Identifica el **equipo/categoría específica** del jugador. Los 12 valores únicos se corresponden con las categorías formativas y senior de la Real Sociedad y equipos asociados.

### 4.4. `GrupoEdad` — Grupo de edad/categoría del equipo

| Atributo | Valor |
|---|---|
| **Rol** | Variable de agrupación (VI₅) |
| **Tipo** | Categórica nominal |
| **Unidad** | — (categoría) |
| **Fuente** | Derivada — creada en NB03 como mapeo de `NombreCorrecto` |
| **Transformación** | Recodificación de los 12 equipos en 5 grupos de edad/categoría |

**Mapeo aplicado:**

| `NombreCorrecto` | → `GrupoEdad` |
|---|---|
| Neskak A, Neskak B, Neskak C | Neskak |
| Infantil Txiki, Infantil Handi | Infantil |
| Cadete Txiki, Cadete Vasca | Cadete |
| EASO, Juvenil División de Honor | Juvenil |
| Real Sociedad C, SANSE, Real Sociedad | Senior Masculino |

**Nota sobre `Neskak`:**  
Agrupa a los equipos femeninos de la Real Sociedad. Se mantienen como categoría separada porque las diferencias de género en rendimiento físico podrían confundir el efecto de la edad si se mezclaran con los equipos masculinos.

### 4.5. `Phase Duration (min)` — Duración de la fase

| Atributo | Valor |
|---|---|
| **Rol** | Variable auxiliar (denominador de todas las VD normalizadas) |
| **Tipo** | Cuantitativa discreta |
| **Unidad** | minutos |
| **Fuente** | PlayerMaker (raw) |
| **Transformación** | NB01: verificada consistencia intra-Phase Id, 0 filas con duración = 0. NB03: utilizada como denominador para crear las 4 VD normalizadas |

**Papel en el pipeline:**  
Es la variable más transversal del dataset: aparece como denominador en las 4 métricas normalizadas (`Total Touches / min`, `Golpeos +15 m/s / min`, `Distance Covered (m) / min`, `High Intensity Distance (20 km/h) / min`). Su integridad es crítica para la validez de todas las tasas por minuto.

### 4.6. `Position Category` — Categoría de posición

| Atributo | Valor |
|---|---|
| **Rol** | Variable de contexto |
| **Tipo** | Categórica nominal |
| **Unidad** | — (categoría) |
| **Fuente** | PlayerMaker (raw) |
| **Transformación** | Ninguna |

**Interpretación:**  
Categoría general de posición del jugador (ej. Defensa, Centrocampista, Delantero). Registrada directamente por el dispositivo según la configuración del entrenador en la plataforma.

### 4.7. `Position` — Posición específica

| Atributo | Valor |
|---|---|
| **Rol** | Variable de contexto |
| **Tipo** | Categórica nominal |
| **Unidad** | — (categoría) |
| **Fuente** | PlayerMaker (raw) |
| **Transformación** | Ninguna |

**Interpretación:**  
Posición específica del jugador dentro de su categoría (ej. Central, Lateral Derecho, Mediapunta). Mayor granularidad que `Position Category`.

---

## 5. Resumen del diccionario

In [ ]:
# ── Tabla resumen del diccionario ──
variables_resumen = [
    # VD normalizadas
    ("Total Touches / min",                          "VD normalizada", "Toques por minuto"),
    ("Golpeos +15 m/s / min",                        "VD normalizada", "Golpeos potentes por minuto"),
    ("Distance Covered (m) / min",                   "VD normalizada", "Distancia por minuto"),
    ("High Intensity Distance (20 km/h) / min",      "VD normalizada", "Distancia alta intensidad por minuto"),
    # VD absolutas
    ("Total Touches (#)",                            "VD absoluta", "Número total de toques"),
    ("Golpeos +15 m/s",                              "VD absoluta", "Golpeos potentes (conteo)"),
    ("Distance Covered (m)",                         "VD absoluta", "Distancia total recorrida"),
    ("High Intensity Distance (20 km/h)",            "VD absoluta", "Distancia a alta intensidad (> 5.5 m/s ≈ 20 km/h)"),
    # Diseño de tarea
    ("Agrupacion",                                   "VI (tarea)", "Tamaño de agrupación"),
    ("Espacio",                                      "VI (tarea)", "Tipo de espacio"),
    ("Polaridad",                                    "VI (tarea)", "Polaridad de la tarea"),
    ("Equilibrio",                                   "VI (tarea)", "Equilibrio numérico"),
    # Contexto
    ("Phase Id",                                     "ID", "Identificador de fase"),
    ("Player Id",                                    "ID", "Identificador de jugador"),
    ("NombreCorrecto",                               "Contexto", "Nombre del equipo/categoría"),
    ("GrupoEdad",                                    "VI (contexto)", "Grupo de edad del equipo"),
    ("Phase Duration (min)",                         "Auxiliar", "Duración de la fase"),
    ("Position Category",                            "Contexto", "Categoría de posición"),
    ("Position",                                     "Contexto", "Posición específica"),
]

filas = []
for var, rol, desc in variables_resumen:
    if var not in df.columns:
        continue
    s = df[var]
    fila = {"Variable": var, "Rol": rol, "Descripción": desc}
    
    if pd.api.types.is_numeric_dtype(s):
        fila["Tipo"] = "Numérica"
        fila["Rango / Niveles"] = f"[{s.min():.2f} – {s.max():.2f}]"
    else:
        fila["Tipo"] = f"Categórica ({s.nunique()} niv.)"
        fila["Rango / Niveles"] = ", ".join(sorted(s.dropna().unique())[:6])
    
    n_miss = s.isna().sum()
    fila["Missings"] = f"{n_miss} ({n_miss / N * 100:.1f}%)"
    filas.append(fila)

df_resumen = pd.DataFrame(filas)
display(
    df_resumen.style
    .set_caption(f"Diccionario de variables — {len(filas)} variables, {N:,} observaciones")
    .hide(axis="index")
)

Variable,Rol,Descripción,Tipo,Rango / Niveles,Missings
Total Touches / min,VD normalizada,Toques por minuto,Numérica,[0.10 – 14.42],0 (0.0%)
Golpeos +15 m/s / min,VD normalizada,Golpeos potentes por minuto,Numérica,[0.00 – 4.50],0 (0.0%)
Distance Covered (m) / min,VD normalizada,Distancia por minuto,Numérica,[7.70 – 172.00],0 (0.0%)
HID Covered (m) / min,VD normalizada,Distancia alta intensidad por minuto,Numérica,[0.00 – 86.00],0 (0.0%)
Total Touches (#),VD absoluta,Número total de toques,Numérica,[1.00 – 173.00],0 (0.0%)
Golpeos +15 m/s,VD absoluta,Golpeos potentes (conteo),Numérica,[0.00 – 38.00],0 (0.0%)
Distance Covered (m),VD absoluta,Distancia total recorrida,Numérica,[35.00 – 4442.00],0 (0.0%)
HID Covered (m) Zone 1 [> 4(m/s)],VD absoluta,Distancia a alta intensidad,Numérica,[0.00 – 1299.00],0 (0.0%)
Agrupacion,VI (tarea),Tamaño de agrupación,Categórica (2 niv.),"grande, pequeno",0 (0.0%)
Espacio,VI (tarea),Tipo de espacio,Categórica (2 niv.),"amplio, reducido",0 (0.0%)


---

## 6. Trazabilidad de transformaciones

La siguiente tabla resume todas las transformaciones aplicadas a las variables del estudio, incluyendo la variable de origen y el notebook donde se implementó.

| Variable resultante | Variable(s) de origen | Transformación | Notebook |
|---|---|---|---|
| `Total Touches / min` | `Total Touches (#)`, `Phase Duration (min)` | División: `Total Touches (#) / Phase Duration (min)` | NB03 |
| `Golpeos +15 m/s` | `RV Zone 4`, `RV Zone 5`, `RV Zone 6` | Suma: `Zone 4 + Zone 5 + Zone 6` | NB03 |
| `Golpeos +15 m/s / min` | `Golpeos +15 m/s`, `Phase Duration (min)` | División | NB03 |
| `Distance Covered (m) / min` | `Distance Covered (m)`, `Phase Duration (min)` | División | NB03 |
| `High Intensity Distance (20 km/h)` | `SD Covered (m) [> 5.5(m/s)]` | Renombrado | NB03 |
| `High Intensity Distance (20 km/h) / min` | `High Intensity Distance (20 km/h)`, `Phase Duration (min)` | División | NB03 |
| `GrupoEdad` | `NombreCorrecto` | Mapeo de 12 equipos → 5 categorías | NB03 |
| `Agrupacion` (reclasificada) | `Agrupacion` (raw), `Espacio` (raw) | Reclasificación de 7 combinaciones Agrupación×Espacio → 2 niveles (`pequeno`, `grande`); eliminación de 464 filas no clasificables | NB01 |
| `Espacio` (reclasificado) | `Agrupacion` (raw), `Espacio` (raw) | Reclasificación de 7 combinaciones Agrupación×Espacio → 2 niveles (`reducido`, `amplio`) | NB01 |
| `Player Id` (unificado) | `Player Id` (raw) | Unificación de IDs duplicados (28 jugadores, 293 filas) | NB01 |
| `NombreCorrecto` (unificado) | `NombreCorrecto` (raw) | Resolución de nombres tras unificación de IDs | NB01 |
| *(eliminaciones)* | `Total Touches (#)` ∧ `Total Touches / min` | Exclusión de instancias con ambas < p05 (175 filas) | NB04 |

### Columnas eliminadas del dataset original

En NB03 se eliminaron **15 columnas** por redundancia con las VD seleccionadas:
- `Session Id`, `Participated Players`
- `Left Leg Touches (#)`, `Right Leg Touches (#)` — subconjuntos de Total Touches
- `Releases (#)` — redundante con Total Touches (r ≈ 0.88)
- `RV Zone 1–6` — capturadas por Golpeos +15 m/s o redundantes
- `HID Covered (m) Zone 1 [> 4(m/s)]` — descartada en favor de SD Covered (renombrada a High Intensity Distance). Su umbral de 4 m/s (≈ 14.4 km/h) es demasiado bajo para alta intensidad real y presenta mayor correlación con Distance Covered (r ≈ 0.81)
- `Release Velocity Max / Avg (m/s)` — métricas de velocidad puntual, descartadas
- `Release Index Beta — Total` — redundante con Total Touches y Releases (r ≈ 0.96)

### Reducción total del pipeline

| Etapa | Filas | Columnas |
|:---|:---:|:---:|
| Datos originales (`PlayerMaker_Matrix.xlsx`) | 5 321 | 30 |
| Tras NB01 (limpieza de negocio + reclasificación Agrupación×Espacio) → `Matriz_V1.xlsx` | 4 621 | 28 |
| Tras NB03 (métricas derivadas) → `Matriz_V2.xlsx` | 4 621 | 19 |
| Tras NB04 (limpieza estadística) → `Matriz_V3.xlsx` | 4 446 | 19 |